## Fine-Tuning with SFT + LoRA for Qwen

This notebook demonstrates how to perform Supervised Fine-Tuning (SFT) on the Qwen model using Parameter-Efficient Fine-Tuning (PEFT) with LoRA and QLoRA.

### Workflow:
1.  **Setup**: Install required libraries and configure paths.
2.  **Data Preparation**: Load the dataset and format it for SFT. Since full reasoning is not available, we create synthetic completions to teach the model the output format.
3.  **Training**: Fine-tune the model using `SFTTrainer`, 4-bit quantization (QLoRA), and LoRA.
4.  **Merge & Save**: Merge the trained LoRA adapters into the base model and save the final fine-tuned model.
5.  **Inference**: Load the merged model with `vLLM` for high-throughput generation.
6.  **Evaluation**: Score the generated responses against the ground truth answers.

### DSMLP Setup
First, run this in your DSMLP terminal:
```bash
launch-sp26-cuda128.sh -l gpu-class=medium -W CSE151B_SP26_A00 -g 1 -c 8 -m 32 -v a30
```

In [ ]:
!pip install -q "transformers==4.40.1" "datasets==2.19.0" "accelerate==0.29.3" "bitsandbytes==0.43.1" "peft==0.10.0" "trl==0.8.6" "vllm==0.4.1"

In [ ]:
import json
import os
import re
import sys
import gc
import torch
from pathlib import Path
from typing import Optional, List

# ── Configuration ─────────────────────────────────────────────────────────────
GPU_ID             = "0"
BASE_MODEL_ID      = "Qwen/Qwen3-4B-Thinking-2507" # Using the same model as baseline
DATA_PATH          = "../data/public.jsonl"      # Adjusted path relative to notebook
ADAPTER_PATH       = "results/sft_lora_adapter"
MERGED_MODEL_PATH  = "results/qwen_sft_merged"
OUTPUT_PATH        = "results/sft_results.jsonl"
MAX_SEQ_LENGTH     = 4096
MAX_TOKENS_INFER   = 32768 # Max tokens for vLLM generation, matching baseline

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

from datasets import Dataset
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer
from vllm import LLM, SamplingParams
from tqdm import tqdm

In [ ]:
# Check GPU status
!nvidia-smi

## Part 1: Data Preparation for SFT

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

def create_sft_conversations(data: List[dict]) -> List[dict]:
    """Formats raw data into conversations for SFT, including synthetic assistant responses."""
    conversations = []
    for item in data:
        system, user = build_prompt(item["question"], item.get("options"))
        
        # NOTE: The public dataset does not contain step-by-step reasoning.
        # We create a synthetic completion that only includes the final answer in the required format.
        # This fine-tunes the model to follow output formatting instructions.
        if item.get("options"):
            assistant_response = f"The final answer is \\boxed{{{item['answer']}}}."
        else:
            gold_answer = item['answer']
            if isinstance(gold_answer, list):
                gold_answer = gold_answer[0] # Use the first answer for simplicity in training
            assistant_response = f"The final answer is \\boxed{{{gold_answer}}}."

        conversations.append({
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
                {"role": "assistant", "content": assistant_response}
            ]
        })
    return conversations

# Load data and create SFT dataset
raw_data = [json.loads(line) for line in open(DATA_PATH)]
sft_formatted_data = create_sft_conversations(raw_data)
sft_dataset = Dataset.from_list(sft_formatted_data)

print(f"Loaded and formatted {len(sft_dataset)} samples for SFT.")
print("\n── SFT Sample (first 2 messages) ──")
print(json.dumps(sft_dataset[0]['messages'][:2], indent=2))
print("\n── SFT Sample (assistant message) ──")
print(json.dumps(sft_dataset[0]['messages'][2], indent=2))

## Part 2: QLoRA Fine-Tuning

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# QLoRA configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load base model with quantization
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# LoRA configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # For Qwen2, target modules are typically these. Adjust if using a different model.
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

# Training arguments
training_args = TrainingArguments(
    output_dir=ADAPTER_PATH,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    fp16=False, # bf16 is enabled by default with torch_dtype=torch.bfloat16
    bf16=True,
    report_to="none",
)

# Initialize SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=sft_dataset,
    peft_config=peft_config,
    dataset_num_proc=4,
    dataset_format="chatml", # Use 'chatml' for list of message dicts
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    args=training_args,
)

# Start training
print("Starting SFT with QLoRA...")
trainer.train()
trainer.save_model(ADAPTER_PATH)
print(f"Training complete. LoRA adapters saved to {ADAPTER_PATH}")

## Part 3: Merge Adapters and Save Model

In [ ]:
# Clean up memory before loading full model
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

print("Loading base model for merging...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Loading PEFT model from {ADAPTER_PATH}...")
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

print("Merging LoRA adapters...")
merged_model = peft_model.merge_and_unload()

print(f"Saving merged model to {MERGED_MODEL_PATH}...")
merged_model.save_pretrained(MERGED_MODEL_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_MODEL_PATH)

print("Model merging and saving complete.")

## Part 4: Inference with the Fine-Tuned Model

In [ ]:
# Clean up memory again before loading with vLLM
del base_model
del peft_model
del merged_model
gc.collect()
torch.cuda.empty_cache()

print(f"Loading fine-tuned model from {MERGED_MODEL_PATH} with vLLM...")
llm = LLM(
    model=MERGED_MODEL_PATH,
    dtype="bfloat16",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.50, # Matching baseline
    max_model_len=8192,          # Matching baseline
    trust_remote_code=True,
    max_num_seqs=1,              # Matching baseline (NOTE: this will be slow for many prompts)
    max_num_batched_tokens=32768,# Matching baseline
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS_INFER, # Now 32768
    temperature=0.7,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Fine-tuned model loaded for inference.")

In [ ]:
# Build prompts for the entire dataset
prompts = []
for item in raw_data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={raw_data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## Part 5: Scoring and Evaluation

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

# Load Judger for free-form scoring
sys.path.insert(0, "..") # Adjust path to find judger.py
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(raw_data, responses), total=len(raw_data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS (SFT MODEL)")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")